In [712]:
import pandas as pd
import numpy as np
from math import e
import matplotlib.pyplot as plt

pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", None)


column_types = {
    "id": "Int64",
    "osm_id": "str",
    "addr_street": "str",
    "addr_number": "str",
    "ground_floor_group": "str",
    "ground_floor_function": "str",
    "upper_floor_group": "str",
    "upper_floor_function": "str",
    "number_of_storeys": "Int64",
    "is_mixed_use": "boolean",
    "build_year_est": "Int64",
    "arch_style": "str",
    "condition": "Int64",
    "max_height": "float64",
    "is_heritage": "str",
    "source": "str",
    "confidence": "Int64",
    "date_of_checking": "str",
    "number_of_functions": "Int64",
}

In [713]:
data_gdynia_long_categories = pd.read_csv(
    "../data/processed/fixed/srodmiescie_long_categories.csv",
    sep=";",
    index_col=0,
    dtype=column_types,
)
cat_excluded = ["brak parteru", "pustostan"]
data_gdynia_long_categories_wo_excluded = data_gdynia_long_categories.loc[
    ~data_gdynia_long_categories["category"].isin(cat_excluded)
]

In [714]:
top_2_cat_str = (
    data_gdynia_long_categories_wo_excluded.groupby(["addr_street", "category"])
    .size()
    .sort_values(ascending=False)
    .groupby("addr_street")
    .head(2)
)


1.

In [715]:
most_pop_cat = (
    data_gdynia_long_categories_wo_excluded.groupby(["addr_street", "category"])
    .size()
    .groupby("addr_street")
    .idxmax()
    .str[1]
)

In [ ]:
most_pop_cat2 = (
    data_gdynia_long_categories_wo_excluded.groupby(["addr_street", "category"])
    .agg(num_buildings=("category", "size"))
    .assign(num_build_street_max=lambda d: d.groupby("addr_street").transform("max"))
    .assign(is_leader=lambda d: d["num_buildings"] == d["num_build_street_max"])
    .groupby("addr_street")
    .agg(num_max=("is_leader", "sum"))
)
print(f"num of streets with 1st place draw: {most_pop_cat2['num_max'].gt(1).sum()}")
print(
    f"{most_pop_cat2['num_max'].max()}, {most_pop_cat2.loc[most_pop_cat2['num_max'].eq(most_pop_cat2['num_max'].max())].index}"
)

num of streets with 1st place draw: 9
4, Index(['Arkadiusza Rybickiego', 'Żołnierzy I Armii Wojska Polskiego'], dtype='str', name='addr_street')


In [717]:
most_pop_cat.value_counts()[:3]

handel          13
mieszkaniowa     9
gastronomia      8
Name: count, dtype: int64

In [718]:
assert len(most_pop_cat) == 44

2.

In [719]:
top_2_cat_nlarg = (
    data_gdynia_long_categories_wo_excluded.groupby(["addr_street", "category"])
    .size()
    .groupby(by="addr_street", group_keys=False)
    .nlargest(2)
)

In [720]:
assert len(top_2_cat_str) == len(top_2_cat_nlarg) == 83


3.

In [ ]:
rank_cat = (
    data_gdynia_long_categories_wo_excluded.groupby(["addr_street", "category"])
    .agg(num_buildings=("category", "size"))
    .assign(
        rank_street=lambda d: d.groupby("addr_street").rank(
            method="dense", ascending=False
        )
    )
    .loc[lambda d: d["rank_street"].le(2)]
)

In [ ]:
assert len(rank_cat) == 114
assert rank_cat.groupby("addr_street").size().gt(2).sum() == 18

4.

In [ ]:
share_leader = (
    data_gdynia_long_categories_wo_excluded.groupby(["addr_street", "category"])
    .agg(num_buildings=("category", "size"))
    .groupby("addr_street")["num_buildings"]
    .agg(["max", "sum"])
    .assign(share_leader=lambda d: d["max"].div(d["sum"]))
)
share_leader.tail()

,max,sum,share_leader
addr_street,,,
Śledziowa,6,9,0.666667
Świętego Piotra,3,10,0.300000
Świętego Wojciecha,1,3,0.333333
Świętojańska,91,214,0.425234
Żołnierzy I Armii Wojska Polskiego,1,4,0.250000


In [ ]:
assert share_leader["share_leader"].ge(0.5).sum() == 13
assert (share_leader["share_leader"].ge(0.5) & share_leader["sum"].ge(10)).sum() == 2

In [ ]:
share_leader.loc[share_leader["share_leader"].idxmax()]

max             1.0
sum             1.0
share_leader    1.0
Name: Hilarego Jastaka, dtype: float64

In [ ]:
share_leader.loc[share_leader["share_leader"].idxmin()]

max              3.0000
sum             16.0000
share_leader     0.1875
Name: Portowa, dtype: float64

5.

In [ ]:
top_cat_6 = (
    data_gdynia_long_categories_wo_excluded
    .groupby(["addr_street", "category"])
    .size()
    .sort_values(ascending=False)
    .groupby("addr_street")
)
cat_leader_advantage = (
    top_cat_6.nth(0)
    .droplevel("category")
    .sub(top_cat_6.nth(1).droplevel("category"), fill_value=0)
)

In [ ]:
assert len(cat_leader_advantage) == 44
assert cat_leader_advantage.eq(0).sum() == 9
pd.testing.assert_series_equal(
    cat_leader_advantage.eq(0), most_pop_cat2["num_max"].gt(1), check_names=False
)
assert cat_leader_advantage.idxmax() == "Świętojańska"
assert cat_leader_advantage.max() == 38

6.

In [ ]:
num_empty = (
    data_gdynia_long_categories.loc[lambda d: d["category"] == "pustostan"]
    .groupby("addr_street")
    .size()
)

num_missing_0 = (
    data_gdynia_long_categories.loc[lambda d: d["category"] == "brak parteru"]
    .groupby("addr_street")
    .size()
)

pd.concat({"num_empty": num_empty, "num_missing_0": num_missing_0}, axis=1).assign(
    sum=lambda d: d["num_empty"].add(d["num_missing_0"], fill_value=0)
).sort_values("sum", ascending=False)

,num_empty,num_missing_0,sum
addr_street,,,
Świętojańska,14.0,1.0,15.0
Antoniego Abrahama,5.0,3.0,8.0
Armii Krajowej,1.0,4.0,5.0
Jana z Kolna,5.0,NaN,5.0
10 Lutego,3.0,1.0,4.0
Skwer Tadeusza Kościuszki,1.0,2.0,3.0
Arkadiusza Rybickiego,3.0,NaN,3.0
Antoniego Hryniewickiego,3.0,NaN,3.0
Władysława IV,2.0,1.0,3.0


In [731]:
assert num_empty.add(num_missing_0).isna().sum() == 10
assert (
    len(num_empty.add(num_missing_0))
    == len(num_empty.add(num_missing_0, fill_value=0))
    == 19
)
